In [ ]:
from pathlib import Path
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA, FastICA, NMF, DictionaryLearning
from IPython.display import display

import sys
sys.dont_write_bytecode = True

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'margin_dictionary_learning.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        'Place margin_dictionary_learning.py in the current folder or an ancestor.'
    )
sys.path.insert(0, str(ROOT))

from margin_dictionary_learning import MarginDictionaryLearning

RUN_FITS = False  # False: redraw saved results. True: rerun and replace the CSV.

SEED = 0
N_OBSERVATIONS = 400
ANGLES_DEG = (8, 35, 84)
P_TWO = 0.35
COEF_MIN = 0.2
NOISE_SD = 0.02
ALPHA = 0.05
NMF_SEEDS = (0, 1, 2, 3)
NMF_SHOWN = 3

CAP_QUANTILE = 0.85
N_UPDATES = 5000
LEARNING_RATE = 0.01
CODE_TOL = 1e-5
CODE_MAX_ITER = 5000
RECORD_EVERY = 100
DEVICE = 'auto'
OUTPUT_DIR = ROOT / 'geometry_results'
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_PATH = OUTPUT_DIR / 'geometry_scores.csv'
PLOT_SETTINGS = dict(seed=SEED, observations=N_OBSERVATIONS, angles=ANGLES_DEG,
                     p_two=P_TWO, coef_min=COEF_MIN, noise=NOISE_SD, alpha=ALPHA,
                     nmf_seeds=NMF_SEEDS, nmf_shown=NMF_SHOWN, q=CAP_QUANTILE,
                     updates=N_UPDATES, lr=LEARNING_RATE, code_tol=CODE_TOL,
                     code_max_iter=CODE_MAX_ITER)
PLOT_SETTINGS_JSON = json.dumps(PLOT_SETTINGS, sort_keys=True)

# Small matrices benefit from avoiding a large CPU thread pool.
torch.set_num_threads(2)
started = time.perf_counter()
print('Shared-model device:', 'cuda' if DEVICE == 'auto' and torch.cuda.is_available()
      else 'cpu' if DEVICE == 'auto' else DEVICE)

def make_geometry(seed):
    rng = np.random.default_rng(seed)
    theta = np.deg2rad(ANGLES_DEG)
    true_atoms = np.column_stack((np.cos(theta), np.sin(theta)))
    k = len(true_atoms)
    coefficients = np.zeros((N_OBSERVATIONS, k))
    first = rng.integers(k, size=N_OBSERVATIONS)
    coefficients[np.arange(N_OBSERVATIONS), first] = rng.uniform(
        COEF_MIN, 1.0, size=N_OBSERVATIONS)
    two = rng.random(N_OBSERVATIONS) < P_TWO
    second = (first + rng.integers(1, k, size=N_OBSERVATIONS)) % k
    coefficients[np.arange(N_OBSERVATIONS)[two], second[two]] = rng.uniform(
        0.0, 1.0, size=two.sum())
    noisy = coefficients @ true_atoms + NOISE_SD * rng.standard_normal((N_OBSERVATIONS, 2))
    return noisy.clip(min=0), true_atoms, coefficients, int((noisy < 0).sum())


def unit_rows(atoms):
    atoms = np.asarray(atoms, dtype=float)
    return atoms / np.linalg.norm(atoms, axis=1, keepdims=True)


def match_atoms(true_atoms, fitted_atoms, sign_invariant=False):
    cosine = unit_rows(true_atoms) @ unit_rows(fitted_atoms).T
    if sign_invariant:
        cosine = np.abs(cosine)
    rows, columns = linear_sum_assignment(-cosine)
    scores = np.full(len(true_atoms), np.nan)
    scores[rows] = cosine[rows, columns]
    return scores


if RUN_FITS:
    X, true_atoms, true_codes, n_negative = make_geometry(SEED)
    print(f'Data: {X.shape}, generating directions: {ANGLES_DEG} degrees')
    print(f'Noise made {n_negative} of {X.size} coordinates negative; these were set to zero.')

    rng = np.random.default_rng(SEED)
    initial_atoms = unit_rows(X[rng.choice(len(X), size=3, replace=False)])

    pca = PCA(n_components=2).fit(X)
    ica = FastICA(n_components=2, whiten='unit-variance', max_iter=2000,
                  tol=1e-4, random_state=SEED).fit(X)
    nmf_models = {
        seed: NMF(n_components=3, init='random', max_iter=2000,
                  tol=1e-4, random_state=seed).fit(X)
        for seed in NMF_SEEDS
    }
    sklearn_sdl = DictionaryLearning(
        n_components=3, alpha=ALPHA, fit_algorithm='cd',
        transform_algorithm='lasso_cd', transform_alpha=ALPHA,
        positive_code=True, positive_dict=False, max_iter=500, tol=1e-8,
        random_state=SEED, dict_init=initial_atoms.copy(), code_init=np.zeros((len(X), 3)),
    ).fit(X)

    fitted_atoms = {
        'PCA': pca.components_,
        'ICA': ica.mixing_.T,
        **{f'NMF (init {seed})': model.components_ for seed, model in nmf_models.items()},
        'SDL (scikit-learn)': sklearn_sdl.components_,
    }
    print('PCA, ICA, four NMF initializations and scikit-learn SDL fitted.')

    def recovery_metrics(iteration, dictionary):
        return {'cosine': float(match_atoms(true_atoms, dictionary).mean())}


    def quantile_weights(m):
        positive = m[m > 0]
        cap = torch.quantile(positive, CAP_QUANTILE) if positive.numel() else m.new_zeros(())
        quantile_weights.last_cap = cap  # Record the cap used in the final update.
        return m.clamp_min(0).clamp_max(cap)


    models = {}
    for name, weight_fn in (
        ('SDL (ordinary weights)', lambda m: m.clamp_min(0)),
        ('Clipped SDL', quantile_weights),
    ):
        print(f'\n{name}')
        model = MarginDictionaryLearning(
            3, weight_fn=weight_fn, alpha=ALPHA, lr=LEARNING_RATE,
            n_iter=N_UPDATES, code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER,
            dict_init=initial_atoms, random_state=SEED, device=DEVICE,
            record_every=RECORD_EVERY, callback=recovery_metrics,
        ).fit(X)
        models[name] = model
        fitted_atoms[name] = model.components_

    rows = []
    for name, atoms in fitted_atoms.items():
        is_axis = name in ('PCA', 'ICA')
        scores = match_atoms(true_atoms, atoms, sign_invariant=is_axis)
        angles = np.degrees(np.arctan2(atoms[:, 1], atoms[:, 0]))
        if is_axis:
            angles %= 180
        rows.append({
            'method': name,
            'seed': SEED,
            'n_observations': N_OBSERVATIONS,
            'noise_sd': NOISE_SD,
            'alpha': ALPHA if name in models or name == 'SDL (scikit-learn)' else np.nan,
            'cap_quantile': CAP_QUANTILE if name == 'Clipped SDL' else np.nan,
            'final_cap': float(quantile_weights.last_cap) if name == 'Clipped SDL' else np.nan,
            'angles_deg': ', '.join(f'{angle:.2f}' for angle in np.sort(angles)),
            'matched_atoms': int(np.isfinite(scores).sum()),
            **{f'cosine_{angle}deg': score for angle, score in zip(ANGLES_DEG, scores)},
            'mean_all_three': float(scores.mean()) if np.isfinite(scores).all() else np.nan,
        })
    results = pd.DataFrame(rows).set_index('method')
    results.loc['SDL (scikit-learn)', 'iterations'] = sklearn_sdl.n_iter_
    for name, model in models.items():
        results.loc[name, 'iterations'] = model.n_iter_
        for key in ('relative_gap', 'relative_margin_error'):
            results.loc[name, key] = model.final_coding_diagnostics_[key]
    display(results.style.format(precision=4, na_rep='unmatched / not applicable'))
    print('PCA/ICA angles describe axes modulo 180 degrees; all other angles describe signed rays.')

    for name, model in models.items():
        certificate = model.final_coding_diagnostics_
        print(f'{name}: {model.n_iter_:,} updates in {model.training_time_:.1f}s; '
              f'final coding gap {certificate["relative_gap"]:.2e}, '
              f'relative margin error {certificate["relative_margin_error"]:.2e}')
    difference = results.loc['Clipped SDL', 'mean_all_three'] - results.loc[
        'SDL (ordinary weights)', 'mean_all_three']
    print(f'Clipped minus ordinary final matched cosine on this dataset: {difference:+.6f}')

    centers = {'PCA': pca.mean_, 'ICA': ica.mean_}
    results['atoms_json'] = [json.dumps(fitted_atoms[name].tolist()) for name in results.index]
    results['center_json'] = [json.dumps(centers[name].tolist()) if name in centers else ''
                              for name in results.index]
    results['observations_json'] = ''
    results['truth_json'] = ''
    results['plot_settings_json'] = ''
    results.loc['PCA', 'observations_json'] = json.dumps(X.tolist())
    results.loc['PCA', 'truth_json'] = json.dumps(true_atoms.tolist())
    results.loc['PCA', 'plot_settings_json'] = PLOT_SETTINGS_JSON
    temporary = CSV_PATH.with_suffix('.csv.tmp')
    results.to_csv(temporary)
    temporary.replace(CSV_PATH)
else:
    if not CSV_PATH.exists():
        raise FileNotFoundError('No saved results; set RUN_FITS=True to generate them.')
    results = pd.read_csv(CSV_PATH).set_index('method')
    required = {'atoms_json', 'center_json', 'observations_json', 'truth_json', 'plot_settings_json'}
    if not required.issubset(results.columns):
        raise ValueError('This CSV lacks figure data; set RUN_FITS=True to regenerate it.')
    if json.loads(results.loc['PCA', 'plot_settings_json']) != json.loads(PLOT_SETTINGS_JSON):
        raise ValueError('Settings differ from the saved experiment; restore them or set RUN_FITS=True.')
    fitted_atoms = {name: np.asarray(json.loads(row.atoms_json), dtype=float)
                    for name, row in results.iterrows()}
    centers = {name: np.asarray(json.loads(results.loc[name, 'center_json']))
               for name in ('PCA', 'ICA')}
    X = np.asarray(json.loads(results.loc['PCA', 'observations_json']))
    true_atoms = np.asarray(json.loads(results.loc['PCA', 'truth_json']))
    display(results.drop(columns=list(required)))
    print('Saved results loaded; no fits or CSV writes.')

plt.rcParams.update({
    'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix', 'font.size': 8,
    'axes.titlesize': 8, 'axes.linewidth': 0.5,
})
C_EST, C_TRUE, C_DATA = '#1f5fa8', '0.45', '0.78'
panels = [
    ('(a) generating atoms', None), ('(b) PCA, $K=2$', 'PCA'),
    ('(c) ICA, $K=2$', 'ICA'), ('(d) NMF, $K=3$', f'NMF (init {NMF_SHOWN})'),
    ('(e) SDL (scikit-learn), $K=3$', 'SDL (scikit-learn)'),
    ('(f) Clipped SDL, $K=3$', 'Clipped SDL'),
]
fig, axes = plt.subplots(2, 3, figsize=(6.3, 4.0), sharex=True, sharey=True)
for ax, (title, name) in zip(axes.flat, panels):
    ax.scatter(X[:, 0], X[:, 1], s=4, c=C_DATA, linewidths=0, rasterized=True)
    ax.set(aspect='equal', xlim=(-0.3, 1.7), ylim=(-0.3, 1.7),
           xticks=[0, 1], yticks=[0, 1], title=title)
    ax.tick_params(length=2, pad=1.5, labelsize=7)
    ax.spines[['top', 'right']].set_visible(False)
    for k, atom in enumerate(true_atoms):
        if name is None:
            ax.annotate('', xy=atom, xytext=(0, 0), arrowprops=dict(
                arrowstyle='-|>', color=C_TRUE, lw=1.4, shrinkA=0, shrinkB=0))
            ax.text(*(1.14 * atom), f'$s_{k+1}$', color=C_TRUE, ha='center', va='center')
        else:
            ax.plot([0, atom[0]], [0, atom[1]], color=C_TRUE, lw=1.1, alpha=0.6)
    if name is None:
        continue
    for atom in unit_rows(fitted_atoms[name]):
        if name in ('PCA', 'ICA'):
            mean = centers[name]
            start, end, arrow = mean - 0.75 * atom, mean + 0.75 * atom, '<|-|>'
        else:
            start, end, arrow = (0, 0), atom, '-|>'
        ax.annotate('', xy=end, xytext=start, arrowprops=dict(
            arrowstyle=arrow, color=C_EST, lw=1.4, shrinkA=0, shrinkB=0))
    if name in ('PCA', 'ICA'):
        ax.plot(*mean, 'o', ms=3, mfc='w', mec=C_EST, mew=0.8)
fig.subplots_adjust(left=0.05, right=0.99, top=0.94, bottom=0.06, wspace=0.12, hspace=0.22)
fig.savefig(OUTPUT_DIR / 'fig_dl_geometry.pdf', bbox_inches='tight')
plt.show()

print(f'Finished in {time.perf_counter() - started:.1f}s.')
print(f"Saved {OUTPUT_DIR / 'geometry_scores.csv'} and {OUTPUT_DIR / 'fig_dl_geometry.pdf'}.")
